# Book Market Analytics - Books to Scrape

This notebook for Data Science 500 scrapes the first 300 books from https://books.toscrape.com/ and analyses them. Before making live requests, review the site guidance and its robots.txt, then keep the configured delay.

The analysis follows the five-step process in the study guide: question, acquisition, exploration and preparation, modelling, and communication. Required packages: `requests`, `beautifulsoup4`, `pandas`, `numpy`, `matplotlib`, and `scikit-learn`.

The exported CSV contains only Title, Price, Rating (word label), Availability, Genre, Stock Quantity, and Score (numeric star value from 1 to 5). Additional columns remain in the notebook for exploration and modelling.


In [ ]:
from urllib.parse import urljoin
import re
import time

import numpy as np
import pandas as pd
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

BASE_URL = "https://books.toscrape.com/"  # Live site the data is scraped from (replaces the old synthetic CSV file)
REQUEST_DELAY_SECONDS = 0.25
MAX_PRODUCTS = 300  # Only scrape the first 300 books (None would scrape the whole catalogue)
HEADERS = {"User-Agent": "DataScience500StudentExample/1.0 (educational scraping exercise)"}
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


: 

## 1. Obtain the data

The scraper visits every listing page and each product detail page to collect title, category, price, star rating, availability, quantity where shown, description, and product URL. It uses BeautifulSoup selectors and waits between requests. The site is intended as a scraping sandbox, but it states that prices and ratings are randomly assigned and have no real meaning.


In [ ]:
def get_soup(session, url):
    response = session.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    time.sleep(REQUEST_DELAY_SECONDS)
    return response, BeautifulSoup(response.content, "html.parser")

def scrape_catalogue(max_products=None):
    session = requests.Session()
    current_url = BASE_URL  # Start at the books.toscrape.com home page, then follow the 'next' links
    visited_listing_pages = set()
    rows = []
    while current_url and current_url not in visited_listing_pages:
        visited_listing_pages.add(current_url)
        listing_response, listing_soup = get_soup(session, current_url)
        for card in listing_soup.select("article.product_pod"):
            link = card.select_one("h3 a")
            if link is None or not link.get("href"):
                continue
            product_url = urljoin(listing_response.url, link["href"])
            detail_response, detail_soup = get_soup(session, product_url)
            main = detail_soup.select_one(".product_main")
            if main is None:
                continue
            title_node = main.select_one("h1")
            price_node = main.select_one(".price_color")
            rating_node = main.select_one("p.star-rating")
            stock_node = detail_soup.select_one("p.instock.availability, p.outofstock.availability")
            title = title_node.get_text(" ", strip=True) if title_node else ""
            price_text = price_node.get_text(" ", strip=True) if price_node else ""
            rating_word = next((c for c in (rating_node.get("class", []) if rating_node else []) if c in RATING_MAP), "")
            availability_text = stock_node.get_text(" ", strip=True) if stock_node else ""
            quantity_match = re.search(r"\(([\d,]+)\s+available\)", availability_text, re.I)
            stock_quantity = int(quantity_match.group(1).replace(",", "")) if quantity_match else None
            crumbs = detail_soup.select("ul.breadcrumb li a")
            genre = crumbs[-1].get_text(" ", strip=True) if len(crumbs) >= 3 else "Unknown"
            description_node = detail_soup.select_one("#product_description + p")
            description = description_node.get_text(" ", strip=True) if description_node else ""
            rows.append({
                "book_id": f"LIVE-{len(rows)+1:04d}",
                "product_url": product_url,
                "title": title,
                "genre": genre,
                "price_text": price_text,
                "rating_word": rating_word,
                "availability_text": availability_text,
                "stock_quantity": stock_quantity,
                "description": description,
                "data_source": "books.toscrape.com live scrape"
            })
            if max_products is not None and len(rows) >= max_products:
                return pd.DataFrame(rows)
        next_link = listing_soup.select_one("li.next a")
        current_url = urljoin(listing_response.url, next_link["href"]) if next_link else None
    return pd.DataFrame(rows)

raw_df = scrape_catalogue(max_products=MAX_PRODUCTS)  # Scrapes books.toscrape.com (up to 300 books) instead of reading the CSV
print(f"Collected {len(raw_df)} product records from {BASE_URL}")  # Confirms the live scrape size
raw_df.head()


In [ ]:
import numpy, pandas, requests, bs4, matplotlib, sklearn
print("all libraries imported")

## 2. Clean and prepare

This function accepts the raw scraped fields. It standardizes text, parses GBP prices, maps ratings, normalizes stock status and quantity, removes duplicate products, and records basic text-length features. Missing descriptions are retained because they are not needed by this price model.

The exported CSV is limited to seven requested fields: Title, Price, Rating (word label), Availability, Genre, Stock Quantity, and Score (numeric star value from 1 to 5). Other fields stay in the notebook for analysis and modelling.


In [ ]:
def clean_catalogue(raw):
    data = raw.copy()
    data["title"] = data.get("title", pd.Series(index=data.index, dtype="object")).fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    data["genre"] = data.get("genre", pd.Series(index=data.index, dtype="object")).fillna("Unknown").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    price = data["price_gbp"].astype(str) if "price_gbp" in data.columns else data.get("price_text", pd.Series(index=data.index, dtype="object")).astype(str)
    data["price_gbp"] = pd.to_numeric(price.str.replace(r"[^0-9.]", "", regex=True), errors="coerce")
    if "rating_word" not in data.columns:
        data["rating_word"] = ""
    mapped_rating = data["rating_word"].map(RATING_MAP)
    existing_rating = pd.to_numeric(data.get("star_rating", pd.Series(index=data.index, dtype="float64")), errors="coerce")
    data["star_rating"] = existing_rating.fillna(mapped_rating)
    if "stock_status" not in data.columns:
        availability = data.get("availability_text", pd.Series(index=data.index, dtype="object")).fillna("").astype(str).str.lower()
        data["stock_status"] = np.select([availability.str.contains("in stock"), availability.str.contains("out of stock")], ["In stock", "Out of stock"], default="Unknown")
    data["stock_status"] = data["stock_status"].fillna("Unknown").astype(str).str.strip().str.title()
    quantity = pd.to_numeric(data.get("stock_quantity", pd.Series(index=data.index, dtype="float64")), errors="coerce")
    if "availability_text" in data.columns:
        extracted = data["availability_text"].fillna("").astype(str).str.extract(r"\(([\d,]+)\s+available\)", flags=re.I)[0].str.replace(",", "", regex=False)
        quantity = quantity.fillna(pd.to_numeric(extracted, errors="coerce"))
    data["stock_quantity"] = quantity.mask(data["stock_status"].eq("Out Of Stock"), 0)
    if "description" not in data.columns:
        data["description"] = ""
    data["description"] = data["description"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    data["title_word_count"] = data["title"].str.split().str.len()
    data["description_word_count"] = data["description"].str.split().str.len()
    data = data.dropna(subset=["title", "price_gbp", "star_rating"])
    data = data[data["title"].ne("")].copy()
    if "product_url" in data.columns:
        data = data.drop_duplicates(subset=["product_url"], keep="first")
    else:
        data = data.drop_duplicates(subset=["title", "genre"], keep="first")
    if "book_id" not in data.columns:
        data["book_id"] = [f"EX-{i:04d}" for i in range(1, len(data) + 1)]
    return data.reset_index(drop=True)

books = clean_catalogue(raw_df)
print("Cleaned rows:", len(books))
print("Missing values by column:")
print(books.isna().sum())
print("Duplicate product URLs:", books["product_url"].duplicated().sum() if "product_url" in books.columns else "not available")
export_df = books[["title", "price_gbp", "rating_word", "stock_status", "genre", "stock_quantity", "star_rating"]].rename(columns={
    "title": "Title",
    "price_gbp": "Price",
    "rating_word": "Rating",
    "stock_status": "Availability",
    "genre": "Genre",
    "stock_quantity": "Stock Quantity",
    "star_rating": "Score",
})
export_df = export_df[["Title", "Price", "Rating", "Availability", "Genre", "Stock Quantity", "Score"]]
export_df["Price"] = pd.to_numeric(export_df["Price"], errors="raise")
export_df["Score"] = pd.to_numeric(export_df["Score"], errors="raise").astype("int64")
export_df["Stock Quantity"] = pd.to_numeric(export_df["Stock Quantity"], errors="coerce").astype("Int64")
output_path = Path.cwd() / "sorted books.csv"
export_df.to_csv(output_path, index=False)  # Save only the requested cleaned fields
print(f"Saved cleaned dataset to: {output_path.resolve()}")
export_df.head()

## 3. Explore the catalogue

Use the plots and summaries to describe distributions and relationships. Interpret them as properties of the sample only.


In [ ]:
print(books[["price_gbp", "star_rating", "stock_quantity"]].describe().round(2))
print("\nRows by genre:")
print(books["genre"].value_counts())
print("\nStock status:")
print(books["stock_status"].value_counts(normalize=True).mul(100).round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
books["price_gbp"].plot(kind="hist", bins=15, ax=axes[0], color="#2678A5", edgecolor="white")
axes[0].set_title("Price distribution - live scrape")
axes[0].set_xlabel("Price (GBP)")
axes[0].set_ylabel("Number of books")
axes[0].grid(axis="y", alpha=0.25)
books.groupby("genre")["price_gbp"].mean().sort_values().plot(kind="barh", ax=axes[1], color="#3892BD")
axes[1].set_title("Average price by genre")
axes[1].set_xlabel("Mean price (GBP)")
axes[1].set_ylabel("Genre")
axes[1].grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
books.boxplot(column="price_gbp", by="star_rating", ax=ax, grid=False)
ax.set_title("Price by star rating - live scrape")
ax.set_xlabel("Star rating")
ax.set_ylabel("Price (GBP)")
plt.suptitle("")
plt.tight_layout()
plt.show()
print("Price/rating correlation:", round(books[["price_gbp", "star_rating"]].corr().iloc[0, 1], 3))


## 4. Fit and evaluate linear regression

Price is the response. Genre and stock status are categorical predictors; star rating and stock quantity are numeric predictors. Compare the held-out test set with a training-mean baseline. MAE and RMSE are in GBP; R-squared can be negative when predictions are worse than the test-set mean.


In [ ]:
feature_columns = ["genre", "stock_status", "star_rating", "stock_quantity"]
X = books[feature_columns]
y = books["price_gbp"]
categorical = ["genre", "stock_status"]
numeric = ["star_rating", "stock_quantity"]
preprocess = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("numeric", SimpleImputer(strategy="median"), numeric),
])
model = Pipeline([("preprocess", preprocess), ("regression", LinearRegression())])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
model.fit(X_train, y_train)
predicted = model.predict(X_test)
baseline_prediction = np.repeat(y_train.mean(), len(y_test))

def score_row(name, actual, prediction):
    return {"model": name, "MAE_GBP": mean_absolute_error(actual, prediction),
            "RMSE_GBP": mean_squared_error(actual, prediction) ** 0.5,
            "R_squared": r2_score(actual, prediction)}
metrics = pd.DataFrame([score_row("Linear regression", y_test, predicted),
                        score_row("Training-mean baseline", y_test, baseline_prediction)])
metrics.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, predicted, alpha=0.65, color="#2678A5", edgecolor="white")
limits = [min(y_test.min(), predicted.min()), max(y_test.max(), predicted.max())]
ax.plot(limits, limits, linestyle="--", color="#D35400", label="Perfect prediction")
ax.set_xlim(limits)
ax.set_ylim(limits)
ax.set_xlabel("Observed price (GBP)")
ax.set_ylabel("Predicted price (GBP)")
ax.set_title("Observed and predicted prices")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 5. Communicate the result

Check whether regression improves on the training-mean baseline. Explain MAE in pounds, the meaning of R-squared, and the sample limitations. Save and submit the cleaned dataset, preserve notebook outputs, and report the actual scrape size and date. The website’s warning means its random prices and ratings must not be presented as genuine market evidence.

### Conclusion

The saved scrape contains 300 books collected from the first 15 catalogue pages (23 October 2026). Listed prices ranged from £10.16 to £59.64, with a mean of £34.26 and a median of £35.52, across 40 genres. All 300 books were marked as in stock, so stock status did not vary in this sample.

Using an 80/20 train-test split with random_state=42, the linear regression model had a test MAE of £14.13, RMSE of £16.05 and R-squared of -0.388. The training-mean baseline performed better, with an MAE of £11.53, RMSE of £13.63 and R-squared of -0.002. On average, the regression predictions were about £14.13 away from the listed prices. Its negative R-squared indicates that it performed worse than a simple mean-based prediction on this test set; therefore, these predictors did not produce a useful price model for this sample.

These findings apply only to this 300-book scrape. Books to Scrape identifies itself as a demo site and states that its prices and ratings are randomly assigned, so the observed patterns and model results are for practising data-science methods and are not evidence about real book-market prices. A representative real-world dataset and further validation would be needed for real pricing conclusions.
